# Exercise 2: Open Model + RAG vs. GPT-4o Mini

Compare **Qwen 2.5 1.5B + RAG** against **GPT-4o Mini (no tools)** on the same queries.

**Goal:** Determine whether a larger model without RAG beats a small model with RAG.


In [1]:
# Install required packages
try:
    ip = get_ipython()
    ip.run_line_magic('pip', 'install -q torch transformers sentence-transformers faiss-cpu pymupdf accelerate')
except NameError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch', 'transformers', 'sentence-transformers', 'faiss-cpu', 'pymupdf', 'accelerate'])


In [2]:
import os, sys
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
from typing import List, Tuple
from pathlib import Path

def detect_environment():
    try:
        import google.colab
        return 'colab'
    except ImportError:
        return 'local'

def get_device():
    if torch.cuda.is_available():
        device, dtype = 'cuda', torch.float16
        print(f"✓ CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
    elif torch.backends.mps.is_available():
        device, dtype = 'mps', torch.float32
        print("✓ Apple Silicon MPS")
    else:
        device, dtype = 'cpu', torch.float32
        print("⚠ CPU only")
    return device, dtype

ENVIRONMENT = detect_environment()
DEVICE, DTYPE = get_device()
print(f"Environment: {ENVIRONMENT.upper()} | Device: {DEVICE} | Dtype: {DTYPE}")


✓ CUDA: Tesla T4 (15.6 GB)
Environment: COLAB | Device: cuda | Dtype: torch.float16


In [3]:
from pathlib import Path

DRIVE_BASE = '/content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora'
# Subfolders that will be loaded: ['ModelTService', 'Congressional_Record_Jan_2026']

if ENVIRONMENT == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DOC_FOLDER = DRIVE_BASE
else:
    DOC_FOLDER = str(Path("Corpora"))

print(f"DOC_FOLDER = {DOC_FOLDER}")
assert Path(DOC_FOLDER).exists(), f"Folder not found: {DOC_FOLDER}"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DOC_FOLDER = /content/drive/MyDrive/Colab_Projects/Week05-RAG/Corpora


In [4]:
import fitz

def load_text_file(fp):
    with open(fp, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def load_pdf_file(fp):
    doc = fitz.open(fp)
    parts = []
    for i, page in enumerate(doc):
        t = page.get_text()
        if t.strip():
            parts.append(f"\n[Page {i+1}]\n{t}")
    doc.close()
    return "\n".join(parts)

def load_documents(folder):
    docs = []
    for fp in Path(folder).rglob("*"):
        if not fp.is_file(): continue
        if fp.suffix.lower() not in ('.pdf', '.txt', '.md'): continue
        try:
            content = load_pdf_file(str(fp)) if fp.suffix.lower() == '.pdf' else load_text_file(str(fp))
            if content.strip():
                docs.append((fp.name, content))
                print(f"✓ {fp.name} ({len(content):,} chars)")
        except Exception as e:
            print(f"✗ {fp.name}: {e}")
    return docs

documents = load_documents(DOC_FOLDER)
print(f"\nLoaded {len(documents)} documents")


✓ ModelTNew.pdf (469,891 chars)
✓ ModelTNew.txt (545,492 chars)
✓ EU_AI_Act.pdf (601,538 chars)
✓ EU_AI_Act.txt (597,311 chars)
✓ Ford-Model-T-Man-1919.txt (95,574 chars)
✓ ModelT-01-10.txt (18,676 chars)
✓ ModelT-11-20.txt (19,009 chars)
✓ ModelT-21-30.txt (17,050 chars)
✓ ModelT-31-40.txt (12,194 chars)
✓ ModelT-41-50.txt (14,264 chars)
✓ ModelT-51-60.txt (14,168 chars)
✓ ModelT-61-62.txt (201 chars)
✓ Ford-Model-T-Man-1919-ocr.pdf (95,517 chars)
✓ ModelT-01-10-ocr.pdf (18,658 chars)
✓ ModelT-11-20-ocr.pdf (19,003 chars)
✓ ModelT-21-30-ocr.pdf (17,025 chars)
✓ ModelT-31-40-ocr.pdf (12,201 chars)
✓ ModelT-41-50-ocr.pdf (14,270 chars)
✓ ModelT-51-60-ocr.pdf (14,107 chars)
✓ ModelT-61-62-ocr.pdf (204 chars)
✓ CREC-2026-01-03-v171.pdf (21,119 chars)
✓ CREC-2026-01-29.pdf (428,669 chars)
✓ CREC-2026-01-16.pdf (76,821 chars)
✓ CREC-2026-01-05.pdf (161,797 chars)
✓ CREC-2026-01-06.pdf (1,021,965 chars)
✓ CREC-2026-01-07.pdf (707,729 chars)
✓ CREC-2026-01-08.pdf (1,357,469 chars)
✓ CREC-2026

In [5]:
from dataclasses import dataclass

@dataclass
class Chunk:
    text: str
    source_file: str
    chunk_index: int
    start_char: int
    end_char: int

def chunk_text(text, source_file, chunk_size=512, chunk_overlap=128):
    chunks, start, idx = [], 0, 0
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            pb = text.rfind('\n\n', start + chunk_size // 2, end)
            if pb != -1:
                end = pb + 2
            else:
                sb = text.rfind('. ', start + chunk_size // 2, end)
                if sb != -1:
                    end = sb + 2
        s = text[start:end].strip()
        if s:
            chunks.append(Chunk(s, source_file, idx, start, end))
            idx += 1
        prev_start = start
        start = end - chunk_overlap
        if chunks and start <= chunks[-1].start_char:
            start = end
    return chunks

# Default chunking parameters (override per exercise)
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128

all_chunks = []
for fname, content in documents:
    dc = chunk_text(content, fname, CHUNK_SIZE, CHUNK_OVERLAP)
    all_chunks.extend(dc)
    print(f"  {fname}: {len(dc)} chunks")
print(f"\nTotal chunks: {len(all_chunks)}")


  ModelTNew.pdf: 1496 chunks
  ModelTNew.txt: 1781 chunks
  EU_AI_Act.pdf: 1802 chunks
  EU_AI_Act.txt: 1851 chunks
  Ford-Model-T-Man-1919.txt: 326 chunks
  ModelT-01-10.txt: 64 chunks
  ModelT-11-20.txt: 66 chunks
  ModelT-21-30.txt: 56 chunks
  ModelT-31-40.txt: 44 chunks
  ModelT-41-50.txt: 51 chunks
  ModelT-51-60.txt: 46 chunks
  ModelT-61-62.txt: 1 chunks
  Ford-Model-T-Man-1919-ocr.pdf: 316 chunks
  ModelT-01-10-ocr.pdf: 63 chunks
  ModelT-11-20-ocr.pdf: 61 chunks
  ModelT-21-30-ocr.pdf: 56 chunks
  ModelT-31-40-ocr.pdf: 44 chunks
  ModelT-41-50-ocr.pdf: 47 chunks
  ModelT-51-60-ocr.pdf: 44 chunks
  ModelT-61-62-ocr.pdf: 1 chunks
  CREC-2026-01-03-v171.pdf: 65 chunks
  CREC-2026-01-29.pdf: 1323 chunks
  CREC-2026-01-16.pdf: 254 chunks
  CREC-2026-01-05.pdf: 494 chunks
  CREC-2026-01-06.pdf: 3044 chunks
  CREC-2026-01-07.pdf: 2280 chunks
  CREC-2026-01-08.pdf: 4206 chunks
  CREC-2026-01-08-bk2.pdf: 92 chunks
  CREC-2026-01-08-bk3.pdf: 3597 chunks
  CREC-2026-01-09.pdf: 754 chunk

In [6]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
print(f"Loading: {EMBEDDING_MODEL} on {DEVICE}")
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
EMBEDDING_DIM = embed_model.get_sentence_embedding_dimension()
print(f"Embedding dim: {EMBEDDING_DIM}")


In [7]:
import faiss, pickle
from pathlib import Path

# Cache config — CACHE_KEY encodes corpus + chunk params
# Change it if you change CORPUS_SUBFOLDER, CHUNK_SIZE, or CHUNK_OVERLAP
CACHE_DIR   = Path("/content/drive/MyDrive/Colab_Projects/Week05-RAG/cache")
CACHE_KEY   = "combined_512_128"
CHUNKS_FILE = CACHE_DIR / f"{CACHE_KEY}.chunks.pkl"
EMBEDS_FILE = CACHE_DIR / f"{CACHE_KEY}.embeddings.npy"
INDEX_FILE  = CACHE_DIR / f"{CACHE_KEY}.faiss"
CACHE_DIR.mkdir(exist_ok=True)

def save_cache():
    with open(CHUNKS_FILE, "wb") as f: pickle.dump(all_chunks, f)
    np.save(str(EMBEDS_FILE), chunk_embeddings)
    faiss.write_index(index, str(INDEX_FILE))
    print(f"✓ Cache saved → {CACHE_DIR}/{CACHE_KEY}.*")

def load_cache():
    global all_chunks, chunk_embeddings, index
    with open(CHUNKS_FILE, "rb") as f: all_chunks = pickle.load(f)
    chunk_embeddings = np.load(str(EMBEDS_FILE))
    index = faiss.read_index(str(INDEX_FILE))
    print(f"✓ Cache loaded: {len(all_chunks)} chunks, {index.ntotal} vectors")

if CHUNKS_FILE.exists() and EMBEDS_FILE.exists() and INDEX_FILE.exists():
    load_cache()
else:
    print("No cache found — embedding chunks (first run only, will be cached after)...")
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    faiss.normalize_L2(chunk_embeddings)
    index.add(chunk_embeddings)
    print(f"✓ Index built: {index.ntotal} vectors")
    save_cache()

def rebuild_pipeline(chunk_size=512, chunk_overlap=128):
    """Re-chunk, re-embed, rebuild index in-memory (for Ex 7 & 8 experiments)."""
    global all_chunks, chunk_embeddings, index
    all_chunks = []
    for fname, content in documents:
        all_chunks.extend(chunk_text(content, fname, chunk_size, chunk_overlap))
    chunk_embeddings = embed_model.encode(
        [c.text for c in all_chunks], show_progress_bar=True
    ).astype("float32")
    faiss.normalize_L2(chunk_embeddings)
    index = faiss.IndexFlatIP(EMBEDDING_DIM)
    index.add(chunk_embeddings)
    print(f"Rebuilt: {len(all_chunks)} chunks | size={chunk_size}, overlap={chunk_overlap}")

def retrieve(query, top_k=5):
    qe = embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(qe)
    scores, idxs = index.search(qe, top_k)
    return [(all_chunks[i], float(s)) for s, i in zip(scores[0], idxs[0]) if i != -1]


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading LLM: {LLM_MODEL} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

if DEVICE == 'cuda':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, device_map="auto",
                torch_dtype=DTYPE, trust_remote_code=True)
elif DEVICE == 'mps':
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True).to(DEVICE)
else:
    model = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=DTYPE,
                trust_remote_code=True)
print("LLM loaded ✓")


Loading LLM: Qwen/Qwen2.5-1.5B-Instruct on cuda...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded ✓


In [9]:
PROMPT_TEMPLATE = """You are a helpful assistant that answers questions based on the provided context.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer ONLY based on the information in the context above.
- If the context doesn't contain the answer, say so explicitly.
- Quote relevant passages to support your answer.
- Be concise and direct.

ANSWER:"""

def generate_response(prompt, max_new_tokens=512, temperature=0.3):
    inputs = tokenizer(prompt, return_tensors="pt")
    if DEVICE == 'cuda':
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
    else:
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature,
                             do_sample=(temperature > 0),
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def direct_query(question, max_new_tokens=512):
    prompt = f"Answer this question:\n{question}\n\nAnswer:"
    return generate_response(prompt, max_new_tokens)

def rag_query(question, top_k=5, show_context=False, prompt_template=None):
    results = retrieve(question, top_k)
    ctx_parts = [f"[Source: {c.source_file}, Score: {s:.3f}]\n{c.text}" for c, s in results]
    context = "\n\n---\n\n".join(ctx_parts)
    if show_context:
        print("=== RETRIEVED CONTEXT ==="); print(context); print("=" * 40)
    tmpl = prompt_template if prompt_template else PROMPT_TEMPLATE
    return generate_response(tmpl.format(context=context, question=question))


In [22]:
# Install openai and python-dotenv if not present
try:
    import openai
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'openai'])
    import openai

try:
    from dotenv import load_dotenv
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'python-dotenv'])
    from dotenv import load_dotenv

import os
from pathlib import Path

# Resolve .env path based on environment
if ENVIRONMENT == 'colab':
    env_path = Path('/content/drive/MyDrive/Colab_Projects/Week05-RAG/.env')
else:
    env_path = Path('.env')

load_dotenv(dotenv_path=env_path)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError(f".env file not found or OPENAI_API_KEY missing. Looked at: {env_path}")

client = openai.OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client initialized ✓")

def gpt4o_mini_query(question):
    """Query GPT-4o Mini with no tools or context — pure parametric knowledge."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
        temperature=0.3,
    )
    return resp.choices[0].message.content.strip()

OpenAI client initialized ✓


In [23]:
QUERIES_MODEL_T = [
    "How do I adjust the carburetor on a Model T?",
    "What is the correct spark plug gap for a Model T Ford?",
    "How do I fix a slipping transmission band?",
    "What oil should I use in a Model T engine?",
]
QUERIES_CR = [
    "What did Mr. Flood have to say about Mayor David Black in Congress on January 13, 2026?",
    "What mistake did Elise Stefanik make in Congress on January 23, 2026?",
    "What is the purpose of the Main Street Parity Act?",
    "Who in Congress has spoken for and against funding of pregnancy centers?",
]

ALL_QUERIES = QUERIES_MODEL_T + QUERIES_CR

import time

for q in ALL_QUERIES:
    print(f"\n{'='*70}")
    print(f"QUESTION: {q}")
    print("─" * 70)

    print("[Qwen 2.5 1.5B + RAG]")
    t0 = time.time()
    ans_rag = rag_query(q, top_k=5)
    print(f"{ans_rag}\n  (latency: {time.time()-t0:.1f}s)")

    print("\n[GPT-4o Mini — no tools]")
    t0 = time.time()
    ans_gpt = gpt4o_mini_query(q)
    print(f"{ans_gpt}\n  (latency: {time.time()-t0:.1f}s)")



QUESTION: How do I adjust the carburetor on a Model T?
──────────────────────────────────────────────────────────────────────
[Qwen 2.5 1.5B + RAG]
The instructions for adjusting the carburetor on a Model T include:

1. Inserting an end of a rod through the throttle lever "B" and locking it in place by inserting a cotter pin through its end.
2. Installing a carburetor adjusting rod by inserting its head through a slot in the dash and placing the forked end of the rod "C" through the head of the carburetor needle valve. Lock the rod in position by inserting a cotter key through its end. 

These steps involve manipulating parts of the carburetor assembly to ensure proper adjustment for fuel flow and performance. The context also mentions removing certain components like cotter pins and adjusting rods as part of this process. However, specific details about how these adjustments affect the overall operation or performance of the carburetor are not detailed in the given text.
  (latency: 

## Analysis — Results from Qwen 2.5 1.5B + RAG vs. GPT-4o Mini on Colab T4

> Index: combined corpus (all Corpora/ subfolders), 512-char chunks, 128 overlap.
> GPT-4o Mini training cutoff: October 2023.

### Results Table

| Query | Qwen+RAG | GPT-4o Mini | Verdict |
|-------|----------|-------------|---------|
| **Carburetor adjust** | ✅ Grounded: throttle lever rod, cotter pin, adjusting rod through dash slot, needle valve | ⚠️ Generic modern steps (warm up engine, mixture screw, idle screw) — not Model T-specific | Qwen+RAG wins |
| **Spark plug gap** | ⚠️ Said 7/16" (wrong — again a dimension from a wrong chunk); also mentioned "thickness of a dime" | ✅ Correct: 0.025" (0.635 mm), general guidance to check service manual | GPT-4o Mini wins |
| **Transmission band** | ✅ Grounded: slow-speed band, loosen lock nut, turn adjusting screw, remove cover door | ❌ Generic modern auto transmission advice (ATF fluid, band adjustment screws) | Qwen+RAG wins |
| **Engine oil** | ❌ Cross-corpus: retrieved Learjet "Gates Learjet Airplane Flight Manual for Approved Oils" | ✅ Correct: non-detergent 30W, straight mineral oil, avoid modern multi-viscosity additives | GPT-4o Mini wins |
| **Mr. Flood / Black (CR Jan 13)** | ✅ Correct: Papillion NE, 20+ years service, not seeking reelection, economic growth | ❌ Admits: "training cutoff Oct 2023 — cannot answer Jan 2026 events" | Qwen+RAG wins; GPT-4o Mini correctly declines |
| **Stefanik mistake (CR Jan 23)** | ❌ Fabricated Jan 6 insurrection content; also quoted raw PDF metadata bytes | ✅ Admits: "training cutoff Oct 2023 — cannot answer Jan 2026 events" | GPT-4o Mini wins by admitting ignorance; RAG hallucinated dangerously |
| **Main Street Parity Act (CR Jan 20)** | ✅ Correct: 10% equity requirement, plant acquisition loans, 504 program alignment | ❌ Confabulated a different act entirely (fintech/bank regulatory parity) | Qwen+RAG wins |
| **Pregnancy centers (CR Jan 21)** | ✅ Named Ms. Dexter (OR) and Mr. Schneider with direct quotes (but mislabeled who was for/against) | ⚠️ General: cites Warren, Sanders, Republican leaders — no Jan 2026 specifics | Qwen+RAG wins |

**Score: Qwen+RAG 5 wins · GPT-4o Mini 2 wins · 1 mixed**

---

### Questions Answered

**Does GPT-4o Mini avoid hallucinations better than Qwen 1.5B?**
Partially. GPT-4o Mini avoids hallucinating on post-cutoff CR questions by correctly refusing to answer — a significant safety advantage. But it still confabulated the Main Street Parity Act (described a completely different fintech bill), and gave generic, ungrounded Model T steps. Qwen+RAG hallucinated more dangerously when retrieval failed (fabricating violent Jan 6 events for Stefanik, retrieving Learjet docs for oil), but when the right chunk was retrieved, it was far more specific and accurate than GPT-4o Mini.

**GPT-4o Mini training cutoff vs. corpus age:**

| Corpus | Age | GPT-4o Mini knowledge |
|--------|-----|-----------------------|
| Model T manual | ~1908–1919 | Partial — knows the vehicle but not manual-specific details |
| Congressional Record Jan 2026 | ~27 months post-cutoff | None — correctly refuses to answer |

This is the clearest illustration of when RAG adds value: **for post-cutoff or domain-specific content**, RAG with a 1.5B model outperforms a much larger model with no retrieval. For well-documented historical knowledge (Model T specs like spark plug gap, oil type), GPT-4o Mini's parametric knowledge can match or beat imperfect retrieval.

**Which system wins by query type:**
- **Model T mechanical procedures** (carburetor, transmission): Qwen+RAG wins — retrieves specific part names and sequences
- **Model T numeric specs** (gap, oil weight): GPT-4o Mini wins — specs are in training data; RAG kept grabbing wrong chunks
- **Post-cutoff CR events** (Flood, Parity Act, pregnancy centers): Qwen+RAG wins — GPT-4o Mini has no knowledge
- **Post-cutoff CR edge cases** (Stefanik): Both fail — RAG retrieved wrong content; the short Jan 23 doc (61 chunks) gets drowned out in the 139k-chunk combined index


In [24]:
import pickle

def save_index(path):
    faiss.write_index(index, f"{path}.faiss")
    with open(f"{path}.chunks", 'wb') as f:
        pickle.dump(all_chunks, f)
    print(f"✓ Saved {path}.faiss + .chunks")

save_index("rag_index")


✓ Saved rag_index.faiss + .chunks
